# Lesson 8 — SQL Joins and Analytics
### Intro to Computer Science for Non-CS Students

**The bridge from Lessons 2 and 5.** Two threads you already own meet here:

1. **Lesson 2, Unit 8:** you combined two tables with `merge(how=...)`, and saw a
   join *drop* rows, *keep* rows as `NaN`, and even *multiply* rows. That was
   pandas giving you a verb.
2. **Lesson 5:** you learned to tame a long computation by **naming its pieces** —
   extracting well-named functions so the logic reads top-to-bottom.

SQL is the **declarative twin** of both. A `JOIN` is `merge` written as a
sentence; a `WITH` clause is Lesson 5's extract-a-function move; and the analytics
you did with `groupby` become `GROUP BY`, `CASE WHEN`, and — the new superpower —
**window functions**. This is *the* interview lesson: the patterns below are the
ones you will be asked to write on a whiteboard.

```
Business question              "Top 3 customers in each country?"
        ↓
Tables + keys      (Lesson 2)   customers ⋈ transactions on customer_id
        ↓
Declarative query  (this one)   JOIN · CASE WHEN · WITH · window functions
        ↓
The same answer                 identical to the pandas you already write
```

**The through-line, one more time.** A structure is defined by its *properties*,
and every tool either preserves or loses them. A SQL **table** is the most
property-rich structure in the course: its *schema* pins down each column's type,
and a key can promise uniqueness. This lesson is about the operations those
properties unlock.

**Prerequisites and setup**

- Lesson 1: keys → records (dictionaries), sets and uniqueness, ordering.
- Lesson 2: `merge`, `groupby`, the "reconcile the total" habit.
- Lesson 5: naming intermediate results.
- Uses `pandas` and Python's built-in `sqlite3` — **no installs, no network**.
- Run top to bottom — later cells depend on earlier ones.

**Suggested sessions:** ① Units 1–2 · ② Units 3–4 · ③ Unit 5 + the assignment.

## Setup — a tiny database we can hold in our heads

We use one small SQLite database. It is built from the same pinned UCI Online
Retail extract as Lessons 2 and 3, plus **two clearly-labelled teaching
fixtures** so specific SQL patterns have something real to find:

- **two customers with no transactions** (`C99001`, `C99002`) — so "customers who
  never bought" is a real answer, not an empty set;
- **one exact-duplicate transaction line** — so de-duplication actually bites.

Everything runs against `data/lesson8_retail.sqlite`, which this cell builds. It
is git-ignored and rebuildable — delete it and rerun.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd

print("pandas", pd.__version__, "| SQLite engine", sqlite3.sqlite_version, "(stdlib)")
Path("data").mkdir(exist_ok=True)

In [ ]:
# Build the database from the pinned CSVs + the two documented fixtures.
customers = pd.read_csv("course_data/lesson2_customers_base.csv")          # 10 real customers
transactions = pd.read_csv("course_data/lesson2_transactions_base.csv")    # 60 real lines

# Fixture 1: two authored customers who never place an order.
no_order = pd.DataFrame([
    {"customer_id": "C99001", "customer_label": "Customer 99001",
     "country": "United Kingdom", "first_purchase_date": "2011-02-01"},
    {"customer_id": "C99002", "customer_label": "Customer 99002",
     "country": "EIRE", "first_purchase_date": "2011-03-15"},
])
customers = pd.concat([customers, no_order], ignore_index=True)

# Fixture 2: one exact-duplicate transaction line (the first row, recorded twice).
transactions = pd.concat([transactions, transactions.iloc[[0]]], ignore_index=True)

db_path = Path("data/lesson8_retail.sqlite")
if db_path.exists():
    db_path.unlink()
con = sqlite3.connect(db_path)
customers.to_sql("customers", con, index=False)
transactions.to_sql("transactions", con, index=False)

def run(sql):
    """Run a query and return the result as a DataFrame — our whole interface."""
    return pd.read_sql(sql, con)

print("customers   :", len(customers), "rows")
print("transactions:", len(transactions), "rows  (note: one is a duplicate)")

## Unit 1 — INNER and LEFT JOIN  ·  ~55 min

Two tables, linked by a **key**. `customer_id` lives in both: it is a transaction's
pointer to the customer record — exactly Lesson 1's "a key points to a record"
idea, and exactly Lesson 2's merge key.

```
transactions                      customers
+----------------+-----------+    +-------------+----------------+
| transaction_id | customer  |    | customer_id | country        |
| 536381-01      | C15311 ---+----+-> C15311    | United Kingdom |
+----------------+-----------+    +-------------+----------------+
                    JOIN ... ON transactions.customer_id = customers.customer_id
```

In [ ]:
run("SELECT * FROM customers LIMIT 4")

In [ ]:
run("SELECT * FROM transactions LIMIT 4")

### The schema is where properties live

Unlike a CSV (which forgot every type in Lesson 2), a SQL table carries a
**schema**: the column names *and their types*, declared once and enforced on
every row.

In [ ]:
run("SELECT name, sql FROM sqlite_master WHERE type = 'table'")

🧭 That `sql` column is the table's *definition* — the properties are written
down, not held in our heads. A production schema would go further and declare
`customer_id` a `PRIMARY KEY` (a promise of uniqueness the database enforces).
Types and keys are Lesson 1's properties, finally living *in the structure itself*.

### INNER JOIN — keep only rows that match on both sides

Ask it from the customers' side: *which customers have at least one transaction?*

In [ ]:
run("""
SELECT DISTINCT c.customer_id, c.country
FROM customers c
JOIN transactions t ON c.customer_id = t.customer_id
ORDER BY c.customer_id
""")

Ten customers come back — but we have **twelve**. An INNER JOIN keeps a row
only when the key matches on *both* sides, so the two customers who never bought
silently vanish. That silent drop is exactly what happened to the orphan sale in
Lesson 2.

### LEFT JOIN — keep every row on the left, matched or not

In [ ]:
run("""
SELECT c.customer_id, c.country, COUNT(t.transaction_id) AS n_transactions
FROM customers c
LEFT JOIN transactions t ON c.customer_id = t.customer_id
GROUP BY c.customer_id, c.country
ORDER BY n_transactions
""")

🧭 Twelve rows now — every customer survives, and two show `n_transactions = 0`.
A LEFT JOIN keeps every left-hand row and fills the right-hand columns with **NULL**
where nothing matched. `NULL` is SQL's "there is no value here" — the same hole
`merge` filled with `NaN` in Lesson 2. (`COUNT(t.transaction_id)` counts non-NULL
matches, so the no-order customers score 0.)

### The canonical pattern: LEFT JOIN + IS NULL

"Customers who never bought" = keep everyone, then keep only the rows where the
right side is missing.

In [ ]:
run("""
SELECT c.customer_id, c.country
FROM customers c
LEFT JOIN transactions t ON c.customer_id = t.customer_id
WHERE t.transaction_id IS NULL
""")

🧭 Memorise this shape — it is one of the most-asked interview patterns:
**LEFT JOIN the thing you want to keep, then `WHERE right_key IS NULL` to find the
ones with no match.** Customers who never ordered, products never sold, users who
never logged in — all the same query.

Here is the pandas twin you already wrote in Lesson 2 — same answer, two languages:

In [ ]:
tx = run("SELECT * FROM transactions")
cust = run("SELECT * FROM customers")

left = cust.merge(tx, on="customer_id", how="left", indicator=True)
left.loc[left["_merge"] == "left_only", ["customer_id", "country"]]

🧭 `merge(how="left")` **is** `LEFT JOIN`; the `indicator=True` / `_merge`
trick **is** the `IS NULL` test. SQL just says it as one declarative sentence.

One loose thread before we move on:

In [ ]:
run("SELECT COUNT(*) AS rows, COUNT(DISTINCT transaction_id) AS distinct_ids FROM transactions")

🔮 Sixty-one rows, but only sixty distinct transaction IDs. One line was
recorded twice. Hold that — **Unit 4** cleans it, and shows exactly what it was
quietly costing us.

> **Unit 1 takeaway:** a JOIN matches rows on a key. INNER keeps only matches (and
> can silently drop rows); LEFT keeps every left row and invents `NULL`s for the
> misses. `LEFT JOIN ... WHERE right IS NULL` is the "found nothing on the right"
> pattern.

## Unit 2 — CASE WHEN and conditional aggregation  ·  ~45 min

`CASE WHEN` is SQL's **if/else**, applied to every row — the declarative version of
Lesson 2's `df.loc[cond, col] = ...` and NumPy's `np.where`.

In [ ]:
run("""
SELECT transaction_id, quantity,
       CASE WHEN quantity >= 3 THEN 'bulk' ELSE 'single' END AS order_size
FROM transactions
LIMIT 6
""")

🧭 One expression labels every row — the same "conditional column" you built in
Lesson 2, now written inline in the `SELECT`.

### Conditional aggregation — a pivot in one pass

The trick that surprises people: put a `CASE WHEN` *inside* a `SUM`. Each condition
becomes a count per group.

In [ ]:
run("""
SELECT source_country,
       SUM(CASE WHEN quantity >= 12 THEN 1 ELSE 0 END) AS bulk_lines,
       SUM(CASE WHEN quantity <  12 THEN 1 ELSE 0 END) AS small_lines,
       COUNT(*) AS total_lines
FROM transactions
GROUP BY source_country
""")

🧭 `SUM(CASE WHEN cond THEN 1 ELSE 0 END)` counts the rows meeting `cond`
within each group — so one `GROUP BY` builds a little cross-tab. This is Lesson 2's
`pivot_table` / `crosstab`, expressed as arithmetic. (It is also Lesson 3's
"condition counter" scan, run per group.)

### Segmenting customers by spend band

The everyday version: bucket each customer by how much they spent.

In [ ]:
run("""
SELECT customer_id,
       ROUND(SUM(quantity * unit_price), 2) AS total,
       CASE WHEN SUM(quantity * unit_price) >= 300 THEN 'high'
            WHEN SUM(quantity * unit_price) >= 100 THEN 'mid'
            ELSE 'low' END AS band
FROM transactions
GROUP BY customer_id
ORDER BY total DESC
""")

Now count customers per band. Notice we have to wrap the per-customer query in
another query to group *its* output — a **query inside a query**.

In [ ]:
run("""
SELECT band, COUNT(*) AS n_customers
FROM (
    SELECT customer_id,
           CASE WHEN SUM(quantity * unit_price) >= 300 THEN 'high'
                WHEN SUM(quantity * unit_price) >= 100 THEN 'mid'
                ELSE 'low' END AS band
    FROM transactions
    GROUP BY customer_id
)
GROUP BY band
""")

🔮 That nested `SELECT ( ... )` works, but it reads inside-out and the inner
query has no name. Lesson 5 taught us the fix for exactly this feeling — hold it
for Unit 3.

(One honesty note: these totals still include the duplicated line from Unit 1. It
adds £1.95 to one customer and changes **no** band here — but Unit 4 will remove it
properly and prove the effect.)

> **Unit 2 takeaway:** `CASE WHEN` is vectorised if/else. `SUM(CASE WHEN ...)` turns
> conditions into per-group counts — a pivot table in a single pass.

---
## ⏱️ Session 2 warm-up  ·  5 questions from last time

Answer from memory first, then check.

1. A `JOIN ... ON customer_id` is the declarative twin of which pandas method?
2. A LEFT JOIN keeps rows with no match on the right. What value fills the
   right-hand columns, and what is the one-line pattern to return *exactly* those
   unmatched rows?
3. INNER vs LEFT: which one can silently drop rows, and here — whose rows does it
   drop, customers' or transactions'?
4. `CASE WHEN ... END` plays the role of which everyday programming construct?
5. What does `SUM(CASE WHEN cond THEN 1 ELSE 0 END)` compute, and why is it "a
   pivot in one pass"?

<details><summary><b>Answers</b></summary>

1. `df.merge(..., on="customer_id")`. `ON` names the key both tables share.
2. `NULL` (SQL's "no value"). Pattern: `LEFT JOIN ... WHERE right_key IS NULL`.
3. INNER can drop rows. Here it drops the two *customers* who have no transactions.
4. if/else — a conditional, applied to every row (like `np.where` / `df.loc`).
5. It counts the rows meeting `cond` within each group; several such sums in one
   `GROUP BY` produce a cross-tab (pivot) in a single scan.
</details>

## Unit 3 — CTEs and subqueries  ·  ~45 min

A **CTE** (Common Table Expression), written `WITH name AS (...)`, is the SQL
version of Lesson 5's core move: **give an intermediate result a name** so the rest
of the query reads in plain English. A nested subquery is the anonymous
computation; a CTE is the extracted, well-named function.

Here is Unit 2's band count again — but with the inner query lifted out and
*named*:

In [ ]:
run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(quantity * unit_price) AS total
    FROM transactions
    GROUP BY customer_id
)
SELECT CASE WHEN total >= 300 THEN 'high'
            WHEN total >= 100 THEN 'mid'
            ELSE 'low' END AS band,
       COUNT(*) AS n_customers
FROM customer_totals
GROUP BY band
""")

🧭 Identical answer to the nested version — but now it reads top-to-bottom:
"*first* compute `customer_totals`, *then* band them and count." That is precisely
Lesson 5's refactor: the subquery was an inline lambda; the CTE is a named function.

### A CTE computed once, used twice

Subqueries get painful when you need the same intermediate result more than once —
for example, *customers who spent above the average customer*.

In [ ]:
run("""
WITH customer_totals AS (
    SELECT customer_id, SUM(quantity * unit_price) AS total
    FROM transactions
    GROUP BY customer_id
)
SELECT customer_id, ROUND(total, 2) AS total
FROM customer_totals
WHERE total > (SELECT AVG(total) FROM customer_totals)
ORDER BY total DESC
""")

🧭 `customer_totals` is defined once and referenced **twice** — once in the
`FROM`, once in the `WHERE (SELECT AVG(total) ...)`. Without the CTE you would write
that same aggregation out twice, and any change would have to be made in both
places. This is the Don't-Repeat-Yourself argument from Lesson 5, in SQL.

> **Unit 3 takeaway:** `WITH name AS (...)` names an intermediate table. It turns
> inside-out nested subqueries into a readable top-to-bottom recipe — and lets you
> reuse a result instead of recomputing it. CTEs also *stack*, which is what makes
> the next unit readable.

## Unit 4 — Window functions  ·  ~55 min

The career-making topic, and the second classic interview trap. Here is the whole
idea in one contrast.

A `GROUP BY` **collapses** each group to a single row:

In [ ]:
run("""
SELECT customer_id, ROUND(SUM(quantity * unit_price), 2) AS customer_total
FROM transactions
GROUP BY customer_id
ORDER BY customer_total DESC
LIMIT 4
""")

A **window function** computes across the same group but **keeps every row** —
the per-row detail *and* the group-level number, side by side:

In [ ]:
run("""
SELECT transaction_id, customer_id,
       ROUND(quantity * unit_price, 2) AS line_revenue,
       ROUND(SUM(quantity * unit_price) OVER (PARTITION BY customer_id), 2) AS customer_total
FROM transactions
ORDER BY customer_id
LIMIT 8
""")

🧭 **The trap:** `GROUP BY` answers "one number per group" and *loses the rows*;
a window (`... OVER (PARTITION BY key)`) answers "for each row, a number computed
across its group" and *keeps the rows*. `PARTITION BY` is `GROUP BY`'s window twin.
Interviewers love the difference because reaching for `GROUP BY` when you needed a
window (or vice-versa) is the single most common SQL mistake.

### ROW_NUMBER / RANK — ranking within a partition

Add `ORDER BY` inside the window and you can *number* rows within each group.
Rank each customer within their country by spend:

In [ ]:
run("""
WITH per AS (
    SELECT t.customer_id, c.country,
           SUM(t.quantity * t.unit_price) AS total
    FROM transactions t
    JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY t.customer_id, c.country
)
SELECT country, customer_id, ROUND(total, 2) AS total,
       ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC) AS rn
FROM per
ORDER BY country, rn
""")

### Top-N per group — impossible for plain GROUP BY, trivial for a window

Wrap that in a CTE (they stack, as promised) and filter `rn <= 3`:

In [ ]:
run("""
WITH per AS (
    SELECT t.customer_id, c.country,
           SUM(t.quantity * t.unit_price) AS total
    FROM transactions t
    JOIN customers c ON t.customer_id = c.customer_id
    GROUP BY t.customer_id, c.country
),
ranked AS (
    SELECT country, customer_id, total,
           ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC) AS rn
    FROM per
)
SELECT country, customer_id, ROUND(total, 2) AS total
FROM ranked
WHERE rn <= 3
ORDER BY country, rn
""")

🧭 "Top 3 within each country" cannot be done with a bare `GROUP BY` (it would
collapse the customers away). Rank with a window, then filter on the rank — the
canonical "top-N-per-group" answer. `ROW_NUMBER` always gives distinct numbers;
`RANK` leaves gaps after ties (1, 1, 3); `DENSE_RANK` does not (1, 1, 2) — pick by
how you want ties treated.

### Running totals — a window ordered by time

`ORDER BY` inside the window (without `PARTITION BY`) accumulates over the ordered
rows: a running total.

In [ ]:
run("""
WITH daily AS (
    SELECT transaction_date, SUM(quantity * unit_price) AS revenue
    FROM transactions
    GROUP BY transaction_date
)
SELECT transaction_date,
       ROUND(revenue, 2) AS revenue,
       ROUND(SUM(revenue) OVER (ORDER BY transaction_date), 2) AS running_total
FROM daily
ORDER BY transaction_date
""")

🧭 With `ORDER BY` in the window, `SUM(...) OVER (...)` sums *every row up to and
including this one* — the running total climbing to the grand total on the last row.

### De-duplication with ROW_NUMBER() = 1 — the loose thread from Unit 1

First, *find* the duplicate (this is also an interview pattern):

In [ ]:
run("SELECT transaction_id, COUNT(*) AS copies FROM transactions GROUP BY transaction_id HAVING COUNT(*) > 1")

Now number the copies of each `transaction_id` and keep only the first:

In [ ]:
run("""
WITH numbered AS (
    SELECT transaction_id, customer_id, quantity, unit_price,
           ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY rowid) AS copy_no
    FROM transactions
)
SELECT COUNT(*) AS rows_kept
FROM numbered
WHERE copy_no = 1
""")

Sixty rows kept — the duplicate is gone. And here is what it was quietly
costing us. Reconcile the grand total before and after, the Lesson 2 way:

In [ ]:
run("""
SELECT
    ROUND((SELECT SUM(quantity * unit_price) FROM transactions), 2) AS raw_total,
    ROUND((SELECT SUM(quantity * unit_price) FROM (SELECT DISTINCT * FROM transactions)), 2) AS deduped_total
""")

🧭 The difference is exactly £1.95 — the one duplicated line, double-counted.
**Uniqueness is a property** (Lesson 1); the duplicate silently violated it and
inflated revenue until `ROW_NUMBER() = 1` restored it. This is Lesson 2's
"deduplicate by meaning," now a window function.

> **Unit 4 takeaway:** a window function computes over a group *without collapsing
> the rows* — `PARTITION BY` is the window twin of `GROUP BY`. That unlocks
> ranking (`ROW_NUMBER`/`RANK`), top-N-per-group, running totals, and
> de-duplication (`ROW_NUMBER() = 1`). Reaching for `GROUP BY` when you needed a
> window is the interview's favourite trap.

---
## ⏱️ Session 3 warm-up  ·  5 questions from last time

1. `WITH name AS (...)` is the SQL version of which Lesson 5 refactoring move?
2. `GROUP BY` versus a window function: which one collapses rows, and which keeps
   them?
3. In one line, what shape gives you "the top 3 customers per country"?
4. Why can a plain `GROUP BY` *not* answer "top 3 per country," while a window can?
5. How does `ROW_NUMBER() OVER (PARTITION BY id ORDER BY rowid) = 1` remove exact
   duplicates?

<details><summary><b>Answers</b></summary>

1. Extracting a well-named intermediate (a function) so the logic reads top-to-bottom.
2. `GROUP BY` collapses each group to one row; a window keeps every row and adds a
   computed column.
3. Rank within each country with `ROW_NUMBER() OVER (PARTITION BY country ORDER BY
   total DESC)`, then `WHERE rn <= 3`.
4. `GROUP BY` collapses the customers into a single per-country row, so there is
   nothing left to take the top 3 *of*; a window ranks while keeping each customer.
5. It numbers the copies of each id (1, 2, ...) in row order; keeping only
   `copy_no = 1` keeps one row per id and discards the extra copies.
</details>

## Unit 5 — Interview drill patterns  ·  ~40 min

Most SQL interview questions are a handful of shapes wearing different business
costumes. You have now built every ingredient; here are the recurring shapes,
each stated as the question you will actually be asked, and each solved once.

### "Find the 2nd highest ___." (Nth highest)

Rank descending, keep rank N. `DENSE_RANK` so ties share a place.

In [ ]:
run("""
WITH per AS (
    SELECT customer_id, SUM(quantity * unit_price) AS total
    FROM (SELECT DISTINCT * FROM transactions)
    GROUP BY customer_id
),
ranked AS (SELECT *, DENSE_RANK() OVER (ORDER BY total DESC) AS rnk FROM per)
SELECT customer_id, ROUND(total, 2) AS total
FROM ranked WHERE rnk = 2
""")

🧭 Swap the `2` for any `N`. The classic "2nd highest salary" is this exact
query with a different column name.

### "Find the duplicate records."

`GROUP BY` the key, keep groups with more than one row.

In [ ]:
run("SELECT transaction_id, COUNT(*) AS copies FROM transactions GROUP BY transaction_id HAVING COUNT(*) > 1")

### "Month-over-month change." (compare a row to the previous period)

`LAG` reaches back to the previous ordered row; subtract it.

In [ ]:
run("""
WITH monthly AS (
    SELECT substr(transaction_date, 1, 7) AS month,
           SUM(quantity * unit_price) AS revenue
    FROM (SELECT DISTINCT * FROM transactions)
    GROUP BY month
)
SELECT month, ROUND(revenue, 2) AS revenue,
       ROUND(revenue - LAG(revenue) OVER (ORDER BY month), 2) AS mom_change
FROM monthly
ORDER BY month
""")

🧭 The first month has no previous row, so `LAG` returns `NULL` and the change
is `NULL` — the honest answer, not a fake zero. `LEAD` is the same move looking
*forward*.

### "Gap between consecutive events." (days since the previous one)

`LAG` over each customer's dates; the gap is the difference.

In [ ]:
run("""
WITH dates AS (SELECT DISTINCT customer_id, transaction_date FROM transactions),
gaps AS (
    SELECT customer_id,
           julianday(transaction_date)
             - julianday(LAG(transaction_date) OVER (PARTITION BY customer_id
                                                     ORDER BY transaction_date)) AS gap_days
    FROM dates
)
SELECT customer_id, MAX(gap_days) AS longest_gap_days
FROM gaps
WHERE gap_days IS NOT NULL
GROUP BY customer_id
ORDER BY longest_gap_days DESC
""")

🧭 Only two customers here bought on more than one day, so only they have a
gap — everyone else's `LAG` is `NULL` and drops out. Same pattern powers "days
between logins," "time to repeat purchase," and churn windows.

> **Unit 5 takeaway:** Nth-highest (`DENSE_RANK`), find-duplicates (`GROUP BY ...
> HAVING COUNT > 1`), period-over-period (`LAG`), and event gaps (`LAG` on dates)
> are four shapes that cover a huge fraction of SQL interviews. You built all four
> from the units above.

## Wrap-up

### Six misconceptions this lesson should have broken

| Misconception | Reality |
| --- | --- |
| "A JOIN just glues columns together" | INNER can drop rows, LEFT invents `NULL`s — a join changes the row set |
| "`GROUP BY` and a window do the same thing" | `GROUP BY` collapses rows; a window keeps them — the #1 interview trap |
| "Top-N-per-group needs a loop" | rank with a window, filter the rank — one query, no loop |
| "Nested subqueries are just how SQL looks" | a CTE names the piece and reads top-to-bottom (Lesson 5, in SQL) |
| "A duplicate row is harmless" | it silently double-counted revenue until `ROW_NUMBER() = 1` removed it |
| "SQL and pandas are different skills" | `JOIN`≈`merge`, `GROUP BY`≈`groupby`, `CASE`≈`np.where` — same ideas, two dialects |

### The lesson, one sentence

> *A relational query is `merge` and `groupby` written declaratively — say which
> rows to keep (`JOIN`), how to bucket them (`GROUP BY` / `CASE WHEN`), name the
> pieces (`WITH`), and compute across a group without losing the rows (window
> functions).*

### The cheat sheet — SQL clause → the pandas twin you already know

| SQL | pandas twin | What it does |
| --- | --- | --- |
| `JOIN ... ON k` (INNER) | `merge(on=k, how="inner")` | keep only matched rows |
| `LEFT JOIN ... WHERE r IS NULL` | `merge(how="left", indicator=True)` → `left_only` | rows with no match on the right |
| `CASE WHEN c THEN ... END` | `np.where` / `df.loc[c, col]=` | per-row if/else |
| `SUM(CASE WHEN c THEN 1 ELSE 0 END)` | `pivot_table` / `crosstab` | per-group conditional count |
| `GROUP BY k` + `SUM/COUNT` | `groupby(k).agg(...)` | collapse each group to one row |
| `WITH name AS (...)` | a named intermediate DataFrame | reuse + readability (Lesson 5) |
| `ROW_NUMBER() OVER (PARTITION BY k ORDER BY x)` | `sort_values` + `groupby(k).cumcount()` | rank within a group, keep rows |
| `SUM(x) OVER (ORDER BY t)` | expanding / cumulative sum | running total |
| `LAG(x) OVER (ORDER BY t)` | `shift()` | the previous row's value |

### Where each Lesson 1 property went

- *keys → records (dictionaries)* → the `JOIN ... ON key` that links two tables
- *uniqueness (sets)* → the duplicate that inflated revenue until `ROW_NUMBER = 1`
  restored it; a real `PRIMARY KEY` would have refused it
- *ordering* → `ORDER BY` inside a window is what makes ranking and running totals
  meaningful
- *type attached to the value* → the schema, where a column's type is declared and
  enforced — the property a CSV could not keep

### The assignment — A3: One Analysis, Two Engines

Open **`sql-two-engines-assignment/`**. You will answer the same business
questions **twice** — in SQL and in pandas — against this database, and a grader
checks that both engines agree with a reference *and with each other*. It includes
a benchmark (filter in the database vs drag everything into pandas) and the four
interview drills from Unit 5. Everything in this lesson is a tool you will use
there.

### Extensions

- [Mode SQL tutorial — window functions](https://mode.com/sql-tutorial/sql-window-functions/) — the clearest walk-through of the trap in Unit 4
- [*SQL for Data Scientists*, Renée Teate](https://sqlfordatascientists.com/) — analytics-first SQL, exactly this course's angle
- [SQLite window functions](https://www.sqlite.org/windowfunctions.html) — the reference for the engine you just used
- [pandas — comparison with SQL](https://pandas.pydata.org/docs/getting_started/comparison/comparison_with_sql.html) — every clause, side by side with its pandas twin